# Kraków - oferty mieszkań 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Matix732/wizualizacja-danych-uj/blob/main/Kraków_mieszkania_mapa.ipynb)

Notebook korzysta z pliku `Otodom_Flat_Listings.csv` z repozytorium GitHub.

Zakres:
- filtrujemy oferty do Krakowa,
- wyciągamy dzielnice i poddzielnice oraz cechy mieszkań,
- jeśli w ogłoszeniu brakuje lokalizacji, próbujemy znaleźć dzielnicę po nazwie ulicy,
- robimy mapę dzielnic i wizualizacje danych,

In [ ]:
# Google Colab: instalacja bibliotek
%pip -q install geopandas folium osmnx geopy squarify circlify

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import osmnx as ox
import squarify
import circlify

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Importy gotowe")

Importy gotowe


In [ ]:
# Wczytanie danych z repozytorium GitHub
import urllib.request

csv_url = "https://raw.githubusercontent.com/Matix732/wizualizacja-danych-uj/main/Otodom_Flat_Listings.csv"

try:
    df = pd.read_csv(csv_url)
    print(f"✓ Wczytano z GitHub: {csv_url}")
except Exception as e:
    print(f"✗ Błąd przy pobieraniu z GitHub: {e}")
    raise

print(f"Liczba rekordów (cała Polska): {len(df):,}")

# Normalizacja kolumn
rename_map = {
    "Title": "title",
    "Price": "price",
    "Location": "location",
    "Surface": "surface",
    "Number_of_Rooms": "rooms",
    "Link": "link",
    "City": "city",
}
df = df.rename(columns=rename_map)

for col in ["title", "price", "location", "surface", "rooms", "link", "city"]:
    if col not in df.columns:
        df[col] = np.nan

df["city_norm"] = df["city"].astype(str).str.lower().str.strip()
df["location_norm"] = df["location"].astype(str).str.lower().str.strip()

# Zostawiamy tylko Kraków (na podstawie city lub location)
krk = df[
    df["city_norm"].str.contains("krak", na=False)
    | df["location_norm"].str.contains("krak", na=False)
].copy()

krk["price"] = pd.to_numeric(krk["price"], errors="coerce")
krk["surface"] = pd.to_numeric(krk["surface"], errors="coerce")
krk["rooms"] = pd.to_numeric(krk["rooms"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
krk = krk.dropna(subset=["price"])
krk = krk[krk["price"].between(50_000, 10_000_000)]

print(f"Oferty w Krakowie: {len(krk):,}")
krk[["title", "price", "location", "city"]].head(5)

FileNotFoundError: Nie znaleziono pliku CSV. Umieść go w /content lub katalogu roboczym.

In [ ]:
DISTRICTS = [
    "Stare Miasto", "Grzegórzki", "Prądnik Czerwony", "Prądnik Biały", "Krowodrza",
    "Bronowice", "Zwierzyniec", "Dębniki", "Łagiewniki-Borek Fałęcki", "Swoszowice",
    "Podgórze Duchackie", "Bieżanów-Prokocim", "Podgórze", "Czyżyny", "Mistrzejowice",
    "Bieńczyce", "Wzgórza Krzesławickie", "Nowa Huta"
]

district_keywords = {
    d: [d.lower()] for d in DISTRICTS
}
district_keywords["Stare Miasto"] += ["kazimierz", "stradom", "stare miasto"]
district_keywords["Podgórze"] += ["zabłocie", "podgorze"]
district_keywords["Krowodrza"] += ["azory"]
district_keywords["Dębniki"] += ["ruczaj", "debniki"]
district_keywords["Nowa Huta"] += ["nowa huta"]

def extract_street(text: str):
    if not isinstance(text, str):
        return None
    m = re.search(r"(?:ul\.?|al\.?|os\.?)[\s]+([A-ZĄĆĘŁŃÓŚŹŻa-ząćęłńóśźż0-9\- ]{3,})", text)
    return m.group(0).strip() if m else None

def normalize_tokens(location: str):
    if not isinstance(location, str):
        return []
    return [t.strip() for t in location.split(",") if t.strip()]

def guess_district_from_text(text: str):
    if not isinstance(text, str):
        return None
    txt = text.lower()
    for district, keys in district_keywords.items():
        if any(k in txt for k in keys):
            return district
    return None

def extract_subdistrict(location: str, district: str):
    tokens = normalize_tokens(location)
    if not tokens:
        return None
    cleaned = []
    for t in tokens:
        tl = t.lower()
        if "krak" in tl or "małopol" in tl or tl in {"polska", "poland"}:
            continue
        if district and district.lower() in tl:
            continue
        cleaned.append(t)
    return cleaned[0] if cleaned else None

krk["street"] = krk["location"].apply(extract_street)
krk.loc[krk["street"].isna(), "street"] = krk.loc[krk["street"].isna(), "title"].apply(extract_street)

krk["district"] = krk["location"].apply(guess_district_from_text)
krk["subdistrict"] = krk.apply(lambda r: extract_subdistrict(r["location"], r["district"]), axis=1)

# Wymaganie 4: jeśli brak lokalizacji, użyj ulicy z tytułu i geokoduj do dzielnicy
missing_loc = krk[krk["district"].isna() & krk["street"].notna()].copy()
print(f"Brak dzielnicy po samym location: {len(missing_loc)} rekordów")

if len(missing_loc) > 0:
    geolocator = Nominatim(user_agent="krakow_district_lookup_colab")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

    def district_from_street(street):
        try:
            loc = geocode(f"{street}, Kraków, Polska", addressdetails=True)
            if not loc:
                return None, None
            rev = reverse((loc.latitude, loc.longitude), language="pl", addressdetails=True)
            addr = rev.raw.get("address", {}) if rev else {}
            district_text = " ".join([
                str(addr.get("city_district", "")),
                str(addr.get("suburb", "")),
                str(addr.get("neighbourhood", "")),
            ]).strip()
            district = guess_district_from_text(district_text)
            subdistrict = addr.get("neighbourhood") or addr.get("suburb") or addr.get("quarter")
            return district, subdistrict
        except Exception:
            return None, None

    for idx, row in missing_loc.iterrows():
        d, s = district_from_street(row["street"])
        if d:
            krk.at[idx, "district"] = d
        if s and pd.isna(krk.at[idx, "subdistrict"]):
            krk.at[idx, "subdistrict"] = s

# Domknięcie braków
krk["district"] = krk["district"].fillna("Nieustalona dzielnica")
krk["subdistrict"] = krk["subdistrict"].fillna("Nieustalona poddzielnica")
krk["price_m2"] = (krk["price"] / krk["surface"]).replace([np.inf, -np.inf], np.nan)

krk[["title", "street", "district", "subdistrict", "price", "surface"]].head(10)

In [ ]:
# Granice dzielnic Krakowa (GeoPandas + OSM)
district_gdfs = []
for district in DISTRICTS:
    try:
        q = f"{district}, Kraków, województwo małopolskie, Polska"
        g = ox.geocode_to_gdf(q)
        g["district"] = district
        district_gdfs.append(g[["district", "geometry"]])
    except Exception:
        pass

if not district_gdfs:
    raise RuntimeError("Nie udało się pobrać granic dzielnic z OSM.")

districts_gdf = pd.concat(district_gdfs, ignore_index=True)
districts_gdf = gpd.GeoDataFrame(districts_gdf, geometry="geometry", crs="EPSG:4326")
districts_gdf = districts_gdf.dissolve(by="district", as_index=False)

agg_district = (
    krk.groupby("district", dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
)

districts_plot = districts_gdf.merge(agg_district, on="district", how="left")
districts_plot[["listings", "median_price", "median_price_m2"]] = districts_plot[["listings", "median_price", "median_price_m2"]].fillna(0)

print("Dzielnice z geometrią:", len(districts_plot))
districts_plot.head(3)

In [ ]:
# Mapa dzielnic: liczba ofert (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="listings",
    cmap="YlOrRd",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        ax.text(p.x, p.y, f"{row['district']}\n{int(row['listings'])}", fontsize=7, ha="center")

ax.set_title("Kraków - liczba ofert mieszkań wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.show()

In [ ]:
# Mapa dzielnic: mediana ceny za m2 (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="median_price_m2",
    cmap="RdYlGn_r",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        price_m2 = row['median_price_m2']
        ax.text(p.x, p.y, f"{row['district']}\n{price_m2:,.0f} PLN/m²", fontsize=7, ha="center")

ax.set_title("Kraków - mediana ceny za m² wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.show()


In [ ]:
# Wykres: mediana ceny mieszkań względem mediany metrażu (bez normalizacji)
price_area = (
    krk.dropna(subset=["district", "price", "surface"])
    .groupby("district", as_index=False)
    .agg(
        median_price=("price", "median"),
        median_surface=("surface", "median"),
        listings=("title", "count"),
    )
    .sort_values("median_surface")
)

fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    price_area["median_surface"],
    price_area["median_price"],
    s=price_area["listings"] * 2,
    c=price_area["median_price"],
    cmap="YlOrRd",
    alpha=0.8,
    edgecolors="black",
    linewidth=0.8,
)

for _, row in price_area.iterrows():
    ax.text(
        row["median_surface"] + 0.15,
        row["median_price"],
        row["district"],
        fontsize=8,
        va="center",
    )

ax.set_title("Mediana ceny vs mediana metrażu wg dzielnic", fontsize=14, weight="bold")
ax.set_xlabel("Mediana metrażu [m²]")
ax.set_ylabel("Mediana ceny [PLN]")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}".replace(",", " ")))
ax.grid(alpha=0.3)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Mediana ceny [PLN]")

plt.tight_layout()
plt.show()

In [ ]:
# Styl: Python Graph Gallery - Circular Barplot (mediana ceny po dzielnicach)
circular_data = (
    krk.groupby("district", dropna=False)["price"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_price")
)

N = len(circular_data)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values = circular_data["median_price"].tolist()
labels = circular_data["district"].tolist()

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection="polar"))
ax.bar(angles, values, width=0.5, alpha=0.85, color=sns.color_palette("RdYlGn_r", N))
ax.set_xticks(angles)
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, max(values) * 1.1)
ax.set_title("Circular Barplot: mediana ceny ofert wg dzielnic", fontsize=14, weight="bold", pad=20)
ax.grid(True, alpha=0.3)
plt.show()

# Circular Packing (one level of hierarchy): udział dzielnic w liczbie ofert
packing_data = (
    krk.groupby("district", dropna=False)
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
)

circles = circlify.circlify(
    packing_data["listings"].tolist(),
    show_enclosure=False,
    target_enclosure=circlify.Circle(x=0, y=0, r=1),
)

fig, ax = plt.subplots(figsize=(11, 11))
ax.set_title("Circular Packing: udział dzielnic w liczbie ofert", fontsize=14, weight="bold")
ax.axis("off")
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal")

palette = sns.color_palette("Spectral", len(circles))
total = packing_data["listings"].sum()

for i, (circle, (_, row)) in enumerate(zip(circles, packing_data.iterrows())):
    x, y, r = circle.x, circle.y, circle.r
    share = row["listings"] / total * 100
    patch = plt.Circle((x, y), r, color=palette[i], alpha=0.85, ec="white", lw=1.5)
    ax.add_patch(patch)

    if r > 0.08:
        ax.text(
            x,
            y,
            f"{row['district']}\n{row['listings']}\n{share:.1f}%",
            ha="center",
            va="center",
            fontsize=8,
            weight="bold",
        )

plt.tight_layout()
plt.show()

In [ ]:
# Top oferty i eksport wyników
summary = (
    krk.groupby(["district", "subdistrict"], dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_surface=("surface", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
    .sort_values(["listings", "median_price"], ascending=[False, False])
)

print("Top 20 dzielnica/poddzielnica po liczbie ofert:")
display(summary.head(20))

out_path = Path("/content/krakow_offers_enriched.csv")
krk.to_csv(out_path, index=False)
print(f"Zapisano dane po czyszczeniu: {out_path}")